# 01 · Cleaning, Analysis and Visualizing
**Project:** Predicting *value compression* in a telecom customer base, with a cost-based decision rule
**Notebook 1 of 4** · run order: 01 → 02 → 03 → 04

> **Read this first.** The data was made by `generate_panel_data.py`. Every relationship in it is built into the generator, and it contains no information from any real company. Plots and numbers here show **how the method works**. They do not prove anything about a real customer base.

**What this notebook does**
1. **Cleaning** – load the two tables and check them
2. **Analysis** – define the target ("value compression") and the four customer segments
3. **Visualizing** – compare the segments month by month

**Reads:** `data/panel_monthly.parquet`, `data/customers.parquet`
**Saves:** `work/clean_customers.parquet`, `work/labels.parquet`, `work/figures/*.png`

## 0 · Setup

**Where to run it**
- **Your computer:** keep the folder layout as delivered (`data/` next to the notebooks). Nothing to change.
- **Google Colab:** upload the two parquet files, set `DATA_DIR = "/content"` and `WORK_DIR = "/content/work"`.
- **Kaggle:** add the parquet files as a dataset, set `DATA_DIR = "/kaggle/input/<your-dataset-name>"` and `WORK_DIR = "/kaggle/working"`.

In [ ]:
# ---- Settings: change these two lines if you run on Colab or Kaggle ----
DATA_DIR = "data"   # folder that holds panel_monthly.parquet and customers.parquet
WORK_DIR = "work"   # folder where the notebooks save their results for the next notebook

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

FIG_DIR = os.path.join(WORK_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print("Setup done. Figures will be saved in:", FIG_DIR)

## 1 · Cleaning
**What "cleaning" means here.** We load the data and check it: types, missing values, duplicates, impossible values and business rules. This synthetic data is clean by construction, so the checks confirm it. With real data this section would also fix problems it finds.

### 1.1 Load the two tables
- `panel_monthly` – one row per active customer per month (behaviour and revenue)
- `customers` – one row per customer (static facts, churn status)

In [ ]:
panel = pd.read_parquet(os.path.join(DATA_DIR, "panel_monthly.parquet"))
customers = pd.read_parquet(os.path.join(DATA_DIR, "customers.parquet"))

print("panel     :", panel.shape)
print("customers :", customers.shape)
panel.head()

In [ ]:
customers.head()

### 1.2 Data types and missing values

In [ ]:
print("Column types in the panel:")
print(panel.dtypes)
print()
print("Missing values in the panel:", panel.isna().sum().sum())
print("Missing values in customers:", customers.isna().sum().sum())

### 1.3 Duplicates and impossible values

In [ ]:
print("Duplicate customer-month rows:", panel.duplicated(["customer_id", "month"]).sum())
print("Months in the data:", sorted(panel["month"].unique().tolist()))

values_only = panel.drop(columns=["customer_id", "month"])
print("Negative values in revenue, fees or call counts:", (values_only < 0).sum().sum())
values_only.describe().round(2)

### 1.4 Business rule: a churned customer has no rows after leaving
`churn_month` is the first month a churned customer is gone, so their last row must be the month before.

In [ ]:
check = panel.merge(customers[["customer_id", "churned_flag", "churn_month"]], on="customer_id")
too_late = check[(check["churned_flag"] == 1) & (check["month"] >= check["churn_month"])]
print("Rows on or after a customer's churn month:", len(too_late))
del check, too_late

### 1.5 Missing income and the hidden answer key
- Income uses **-1** to mean "unknown". We keep the -1 (tree models can use it) and add a flag column.
- `customers` also holds `latent_*` columns. These are the generator's **answer key**. They exist only because the data is synthetic; real data would not have them. We remove them from the working copy so they can never leak into a model.

In [ ]:
customers["income_missing"] = (customers["estimated_income_bracket"] == -1).astype(int)
print("Share of customers with unknown income:", round(customers["income_missing"].mean() * 100, 1), "%")

customers_clean = customers.drop(columns=["latent_segment", "latent_event_month"])
customers_clean.to_parquet(os.path.join(WORK_DIR, "clean_customers.parquet"), index=False)
print("Saved clean_customers.parquet with columns:", list(customers_clean.columns))

## 2 · Analysis: define the target
### 2.1 What is "value compression"?
For each customer we take their **last three observed months** and compare the average revenue with the **three months before that**:

`change_rev = (recent average − earlier average) ÷ earlier average × 100`

If `change_rev` is **−25 or lower**, the customer is *compressed* (`value_compressed_flag = 1`).
- Active customers are observed up to month 12.
- Churned customers are observed up to the month before they left.

> Revenue is used **only** to build this label. It is never a model feature, because that would leak the answer.

In [ ]:
# last month each customer was observed
last_month = panel.groupby("customer_id")["month"].max()
last_month.name = "last_month"

panel = panel.join(last_month, on="customer_id")
panel["months_back"] = panel["last_month"] - panel["month"]   # 0 = last month, 1 = the month before, ...
panel[["customer_id", "month", "last_month", "months_back"]].head()

In [ ]:
recent = panel[panel["months_back"] <= 2]
earlier = panel[(panel["months_back"] >= 3) & (panel["months_back"] <= 5)]

recent_avg = recent.groupby("customer_id")["monthly_revenue"].mean()
earlier_avg = earlier.groupby("customer_id")["monthly_revenue"].mean()

change_rev = (recent_avg - earlier_avg) / earlier_avg * 100
change_rev.describe().round(1)

In [ ]:
labels = pd.DataFrame({"change_rev": change_rev, "last_month": last_month})
labels = labels.join(customers_clean.set_index("customer_id")[["churned_flag"]])

labels["value_compressed_flag"] = (labels["change_rev"] <= -25).astype(int)
print("Customers labelled compressed:", labels["value_compressed_flag"].sum())
labels.head()

### 2.2 Four segments
Crossing "active or churned" with "compressed or normal" gives four segments. This is the same comparison used to ask whether customers who are *about to* leave behave like customers who already left.

In [ ]:
status = labels["churned_flag"].map({0: "Active", 1: "Churn"})
spend = labels["value_compressed_flag"].map({0: "Normal", 1: "Compressed"})
labels["segment"] = status + "-" + spend

summary = labels["segment"].value_counts().to_frame("customers")
summary["share %"] = (summary["customers"] / len(labels) * 100).round(1)
summary

In [ ]:
active = labels[labels["churned_flag"] == 0]
rate = active["value_compressed_flag"].mean()

print("Compression rate among active customers:", round(rate * 100, 1), "%")
print("Imbalance (normal : compressed):", round((1 - rate) / rate, 1), ": 1")

In [ ]:
labels.to_parquet(os.path.join(WORK_DIR, "labels.parquet"))
print("Saved labels.parquet")

## 3 · Visualizing
### 3.1 The target: how revenue changed
The red line is the −25 cut-off that defines "compressed".

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(labels["change_rev"].clip(-100, 100), bins=60, color="#1E3A8A", alpha=0.8)
plt.axvline(-25, color="#DC2626", linestyle="--", label="-25% cut-off")
plt.title("Revenue change: last 3 months vs the 3 months before")
plt.xlabel("change_rev (%)  (clipped to -100 ... 100 for the plot)")
plt.ylabel("Customers")
plt.legend()
plt.savefig(os.path.join(FIG_DIR, "target_distribution.png"), dpi=130, bbox_inches="tight")
plt.show()

### 3.2 The four segments

In [ ]:
colors = {"Active-Normal": "#1E3A8A", "Active-Compressed": "#DC2626",
          "Churn-Normal": "#6B7280", "Churn-Compressed": "#B45309"}

counts = labels["segment"].value_counts().reindex(list(colors.keys()))

plt.figure(figsize=(8, 4))
plt.bar(counts.index, counts.values, color=list(colors.values()))
plt.title("Customers per segment")
plt.ylabel("Customers")
plt.xticks(rotation=15)
plt.savefig(os.path.join(FIG_DIR, "segments.png"), dpi=130, bbox_inches="tight")
plt.show()

### 3.3 Behaviour over time, by segment
This is the **timeline comparison**. If customers who are about to leave look like customers who already left, the Active-Compressed and Churn lines should move together, away from Active-Normal.

*What to look for:* which lines rise or fall, and in which months.

In [ ]:
panel = panel.join(labels["segment"], on="customer_id")

columns = ["customer_service_calls", "dropped_blocked_calls", "monthly_overage_fee", "monthly_revenue"]
titles = ["Support calls per month", "Dropped / blocked calls per month",
          "Overage fee (EGP)", "Total revenue (EGP)"]

monthly = panel.groupby(["month", "segment"])[columns].mean().reset_index()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, column, title in zip(axes.flatten(), columns, titles):
    for segment, color in colors.items():
        part = monthly[monthly["segment"] == segment]
        ax.plot(part["month"], part[column], marker="o", color=color, label=segment)
    ax.set_title(title)
    ax.set_xlabel("Month")
axes[0, 0].legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "timeline_by_segment.png"), dpi=130, bbox_inches="tight")
plt.show()

### 3.4 How similar are the segments? (months 7–9)
Each number is the segment's average divided by the Active-Normal average. **1.0 = same as normal customers.**

In [ ]:
behaviour = ["customer_service_calls", "dropped_blocked_calls", "monthly_overage_fee"]

late = panel[panel["month"].between(7, 9)]
level = late.groupby("segment")[behaviour].mean()
ratio = (level / level.loc["Active-Normal"]).round(2)
ratio.loc[list(colors.keys())]

### 3.5 Compression rate by support-call level
Active customers are split into five equal groups by their average monthly support calls (months 1–9).

In [ ]:
early = panel[panel["month"] <= 9]
avg_support = early.groupby("customer_id")["customer_service_calls"].mean()

names = ["Q1 (fewest calls)", "Q2", "Q3", "Q4", "Q5 (most calls)"]
group = pd.qcut(avg_support, 5, labels=names)

active = labels[labels["churned_flag"] == 0]
rate_by_group = active["value_compressed_flag"].groupby(group, observed=True).mean() * 100

plt.figure(figsize=(8, 4))
plt.bar(rate_by_group.index.astype(str), rate_by_group.values, color="#DC2626", alpha=0.85)
plt.title("Share of active customers who became compressed, by support-call level")
plt.ylabel("Compressed (%)")
plt.xticks(rotation=15)
plt.savefig(os.path.join(FIG_DIR, "support_quintiles.png"), dpi=130, bbox_inches="tight")
plt.show()

### 3.6 Do any features duplicate each other?
Two features that are almost identical add no information and can confuse linear models.

In [ ]:
means = early.groupby("customer_id")[["monthly_overage_fee", "data_overage_fee",
                                      "dropped_blocked_calls", "customer_service_calls"]].mean()
means["income_bracket"] = customers_clean.set_index("customer_id")["estimated_income_bracket"]
means["tenure_months"] = early[early["month"] == 9].set_index("customer_id")["account_tenure_months"]

plt.figure(figsize=(7, 5))
sns.heatmap(means.corr(), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Correlation between customer-level features")
plt.savefig(os.path.join(FIG_DIR, "correlation.png"), dpi=130, bbox_inches="tight")
plt.show()

## 4 · Summary
**What the checks showed**
- The data passed every check: no missing values, no duplicates, no negative values, no rows after a customer left.
- Roughly one in ten active customers is labelled *compressed*. That is a clear minority, so plain accuracy would be a poor way to judge a model.

**What the timeline shows**
- Support calls, dropped calls and overage fees rise for the three at-risk segments (Active-Compressed, Churn-Normal, Churn-Compressed) from about month 5, while Active-Normal stays flat.
- In months 7-9 every at-risk segment sits above the normal level on all three behaviours (section 3.4): by roughly 30-80% for support calls and dropped calls. Overage fees differ more between segments, partly because overage is itself part of the revenue that defines the label.
- The share of active customers who become compressed grows steadily with the support-call level (section 3.5).

**Important limit**
The generator gives every at-risk customer the *same* deterioration process, so this similarity is **built in**. On real data the same comparison is the actual test of the idea that customers who are *about to* leave behave like customers who already left.

**Saved for the next notebook:** `work/clean_customers.parquet`, `work/labels.parquet`
**Next:** `02_feature_engineering.ipynb`